# Run FunctionGemma Banking Model

This notebook loads `phattrandeveloper/functiongemma-270m-function-calling` from Hugging Face and runs one end-to-end banking tool-call example.

It uses the same prompt/tokenization path as `fine_tune_function_gemma.ipynb`: `tokenizer.apply_chat_template(..., return_dict=True, return_tensors="pt")`, then `model.generate(...)`.

In [1]:
from __future__ import annotations

import json
import os
import re
import sys
from pathlib import Path
from typing import Any

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "phattrandeveloper/functiongemma-270m-function-calling"
MAX_NEW_TOKENS = 192
SEED = 42

torch.manual_seed(SEED)

def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for path in [current, *current.parents]:
        if (path / "pyproject.toml").exists() and (path / "src" / "tools" / "banking.py").exists():
            return path
    raise RuntimeError("Could not find repo root")

REPO_ROOT = find_repo_root()
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from common import load_config, read_jsonl
from format import format_sample

print("Repo root:", REPO_ROOT)
print("Model:", MODEL_ID)

Repo root: /Users/sonphat.tran/Desktop/Project/fine-tuning-tool-calling
Model: phattrandeveloper/functiongemma-270m-function-calling


In [2]:
CONFIG = load_config("config/settings.yaml")
raw_rows = read_jsonl("data/raw/combined/final_dataset.jsonl")

raw_sample = next(
    row for row in raw_rows
    if row.get("scenario_id") == "recent_transactions_time_period"
)
formatted_sample = format_sample(raw_sample, CONFIG)

# Generate the first assistant turn from only developer + user messages.
messages = formatted_sample["messages"][:2]
tools = formatted_sample["tools"]
expected_first_assistant = formatted_sample["messages"][2]

print("User message:", messages[-1]["content"])
print("Expected first assistant:")
print(json.dumps(expected_first_assistant, ensure_ascii=False, indent=2))

User message: Cho mình xem lịch sử giao dịch 7 ngày gần đây
Expected first assistant:
{
  "role": "assistant",
  "content": null,
  "tool_calls": [
    {
      "type": "function",
      "function": {
        "name": "get_account_info",
        "arguments": {
          "account_id": "ACC_USER",
          "info_type": "transactions",
          "from_date": "2026-05-27",
          "to_date": "2026-06-02",
          "limit": 10
        }
      }
    }
  ]
}


In [ ]:
hf_token = os.getenv("HF_TOKEN")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=hf_token)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=hf_token,
    dtype="auto",
    device_map="auto",
    attn_implementation="eager",
)
model.eval()

print("Loaded model")
print("Device:", model.device)
print("DType:", model.dtype)
print("Special token ids:", {
    "pad": tokenizer.pad_token_id,
    "eos": tokenizer.eos_token_id,
    "bos": tokenizer.bos_token_id,
})

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

Loaded model
Device: mps:0
DType: torch.bfloat16
Special token ids: {'pad': 0, 'eos': 1, 'bos': 2}


In [4]:
START_FUNCTION_CALL = "<start_function_call>"
END_FUNCTION_CALL = "<end_function_call>"
ESCAPE = "<escape>"

def generation_kwargs(max_new_tokens: int = MAX_NEW_TOKENS) -> dict[str, Any]:
    kwargs: dict[str, Any] = {
        "pad_token_id": tokenizer.eos_token_id,
        "max_new_tokens": max_new_tokens,
        "do_sample": False,
    }
    # Guardrail: <pad> is never meaningful assistant output.
    if tokenizer.pad_token_id is not None:
        kwargs["bad_words_ids"] = [[tokenizer.pad_token_id]]
    return kwargs

def generate_next_assistant(
    prefix_messages: list[dict[str, Any]],
    tools: list[dict[str, Any]],
    max_new_tokens: int = MAX_NEW_TOKENS,
) -> dict[str, Any]:
    inputs = tokenizer.apply_chat_template(
        prefix_messages,
        tools=tools,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    )
    with torch.no_grad():
        outputs = model.generate(
            **inputs.to(model.device),
            **generation_kwargs(max_new_tokens),
        )
    generated = outputs[0][len(inputs["input_ids"][0]):]
    token_ids = generated.detach().cpu().tolist()
    pad_count = sum(token_id == tokenizer.pad_token_id for token_id in token_ids)
    text = tokenizer.decode(generated, skip_special_tokens=False)
    return {
        "text": text,
        "token_ids": token_ids,
        "pad_count": pad_count,
        "pad_ratio": pad_count / len(token_ids) if token_ids else 0.0,
        "prompt_tail": tokenizer.decode(inputs["input_ids"][0][-64:], skip_special_tokens=False),
    }

def split_functiongemma_args(argument_text: str) -> list[str]:
    parts: list[str] = []
    buffer: list[str] = []
    in_escape = False
    i = 0
    while i < len(argument_text):
        if argument_text.startswith(ESCAPE, i):
            in_escape = not in_escape
            buffer.append(ESCAPE)
            i += len(ESCAPE)
            continue
        char = argument_text[i]
        if char == "," and not in_escape:
            part = "".join(buffer).strip()
            if part:
                parts.append(part)
            buffer = []
        else:
            buffer.append(char)
        i += 1
    part = "".join(buffer).strip()
    if part:
        parts.append(part)
    return parts

def parse_functiongemma_value(value: str) -> Any:
    value = value.strip()
    if value.startswith(ESCAPE) and value.endswith(ESCAPE):
        return value[len(ESCAPE):-len(ESCAPE)]
    if value == "null":
        return None
    if value == "true":
        return True
    if value == "false":
        return False
    if re.fullmatch(r"-?\d+", value):
        return int(value)
    return value.strip('"')

def parse_tool_call(text: str) -> dict[str, Any]:
    pattern = re.compile(
        re.escape(START_FUNCTION_CALL)
        + r"\s*call:(?P<name>[A-Za-z_]\w*)\{(?P<args>[\s\S]*?)\}\s*"
        + re.escape(END_FUNCTION_CALL)
    )
    match = pattern.search(text)
    if not match:
        return {"type": "text", "content": text.replace("<end_of_turn>", "").replace("<eos>", "").strip()}
    args = {}
    for part in split_functiongemma_args(match.group("args")):
        if ":" not in part:
            continue
        key, value = part.split(":", 1)
        args[key.strip()] = parse_functiongemma_value(value)
    return {"type": "tool_call", "name": match.group("name"), "arguments": args}

def append_assistant_tool_call(messages: list[dict[str, Any]], tool_call: dict[str, Any]) -> None:
    messages.append({
        "role": "assistant",
        "content": None,
        "tool_calls": [{
            "type": "function",
            "function": {
                "name": tool_call["name"],
                "arguments": tool_call["arguments"],
            },
        }],
    })

def append_tool_response(messages: list[dict[str, Any]], name: str, response: dict[str, Any]) -> None:
    messages.append({"role": "tool", "content": {"name": name, "response": response}})

In [5]:
first_generation = generate_next_assistant(messages, tools)
first_action = parse_tool_call(first_generation["text"])

print("--- prompt tail ---")
print(first_generation["prompt_tail"])
print("--- generated first assistant ---")
print(first_generation["text"])
print("--- token summary ---")
print(json.dumps({
    "num_tokens": len(first_generation["token_ids"]),
    "first_token_ids": first_generation["token_ids"][:32],
    "pad_count": first_generation["pad_count"],
    "pad_ratio": first_generation["pad_ratio"],
}, ensure_ascii=False, indent=2))
print("--- parsed action ---")
print(json.dumps(first_action, ensure_ascii=False, indent=2))

--- prompt tail ---
 người nhận.<escape>,type:<escape>STRING<escape>}},required:[<escape>from_account<escape>,<escape>to_account<escape>,<escape>bank_name<escape>,<escape>amount<escape>],type:<escape>OBJECT<escape>}}<end_function_declaration><end_of_turn>
<start_of_turn>user
Cho mình xem lịch sử giao dịch 7 ngày gần đây<end_of_turn>
<start_of_turn>model

--- generated first assistant ---
<start_function_call>call:get_account_info{account_id:<escape>ACC_USER<escape>,from_date:<escape>2026-05-27<escape>,info_type:<escape>transactions<escape>,limit:10,to_date:<escape>2026-06-02<escape>}<end_function_call><start_function_response>
--- token summary ---
{
  "num_tokens": 68,
  "first_token_ids": [
    48,
    6639,
    236787,
    828,
    236779,
    11622,
    236779,
    3906,
    236782,
    11622,
    236779,
    547,
    236787,
    52,
    25458,
    236779,
    20791,
    52,
    236764,
    2543,
    236779,
    1896,
    236787,
    52,
    236778,
    236771,
    236778,
    2368

In [6]:
def execute_demo_tool(name: str, arguments: dict[str, Any]) -> dict[str, Any]:
    if name == "get_account_info":
        return {
            "status": "success",
            "account_id": arguments.get("account_id", "ACC_USER"),
            "transactions": [
                {
                    "date": "2026-06-02",
                    "amount": -250000,
                    "account": "1023456789",
                    "description": "Chuyen khoan tien cafe",
                },
                {
                    "date": "2026-05-31",
                    "amount": 500000,
                    "account": "0909123456",
                    "description": "Nhan tien hoan ung",
                },
            ],
        }
    if name in {"initiate_transfer", "add_beneficiary"}:
        return {"status": "success"}
    if name == "get_beneficiary_info":
        return {"beneficiaries": []}
    return {"status": "error", "message": f"Unknown tool: {name}"}

if first_action["type"] != "tool_call":
    raise RuntimeError(f"Expected a tool call, got: {first_action}")

tool_result = execute_demo_tool(first_action["name"], first_action["arguments"])
append_assistant_tool_call(messages, first_action)
append_tool_response(messages, first_action["name"], tool_result)

print("Tool result:")
print(json.dumps(tool_result, ensure_ascii=False, indent=2))

Tool result:
{
  "status": "success",
  "account_id": "ACC_USER",
  "transactions": [
    {
      "date": "2026-06-02",
      "amount": -250000,
      "account": "1023456789",
      "description": "Chuyen khoan tien cafe"
    },
    {
      "date": "2026-05-31",
      "amount": 500000,
      "account": "0909123456",
      "description": "Nhan tien hoan ung"
    }
  ]
}


In [7]:
final_generation = generate_next_assistant(messages, tools)
final_action = parse_tool_call(final_generation["text"])

print("--- generated final assistant ---")
print(final_generation["text"])
print("--- token summary ---")
print(json.dumps({
    "num_tokens": len(final_generation["token_ids"]),
    "first_token_ids": final_generation["token_ids"][:32],
    "pad_count": final_generation["pad_count"],
    "pad_ratio": final_generation["pad_ratio"],
}, ensure_ascii=False, indent=2))
print("--- parsed final ---")
print(json.dumps(final_action, ensure_ascii=False, indent=2))

--- generated final assistant ---
Đã lấy lịch sử giao dịch theo khoảng thời gian yêu cầu thành công.<end_of_turn>
--- token summary ---
{
  "num_tokens": 17,
  "first_token_ids": [
    237703,
    237029,
    45222,
    57291,
    21994,
    39501,
    32011,
    20107,
    44643,
    21972,
    26404,
    32105,
    32952,
    17419,
    15289,
    236761,
    106
  ],
  "pad_count": 0,
  "pad_ratio": 0.0
}
--- parsed final ---
{
  "type": "text",
  "content": "Đã lấy lịch sử giao dịch theo khoảng thời gian yêu cầu thành công."
}
